# Fraud Detection — Data Preprocessing & Feature Engineering
## IEEE-CIS Dataset | 590,540 Transactions

This notebook handles:
1. Missing value imputation strategy
2. Feature engineering (velocity, time-based, aggregation features)
3. Encoding categorical variables
4. SMOTE oversampling for class balance
5. Train/validation/test split

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded')

## 1. Load & Merge Data

In [ ]:
df_trans = pd.read_csv('../data/raw/train_transaction.csv')
df_id    = pd.read_csv('../data/raw/train_identity.csv')
df = df_trans.merge(df_id, on='TransactionID', how='left')

print(f'Shape after merge: {df.shape}')
print(f'Fraud rate       : {df["isFraud"].mean()*100:.2f}%')

## 2. Feature Engineering — Adding Business-Relevant Features

In [ ]:
def engineer_features(df):
    df = df.copy()

    # --- Time features ---
    df['hour']         = (df['TransactionDT'] // 3600) % 24
    df['day_of_week']  = (df['TransactionDT'] // (3600 * 24)) % 7
    df['is_weekend']   = df['day_of_week'].isin([5, 6]).astype(int)
    df['is_night']     = df['hour'].between(0, 5).astype(int)  # midnight–6am

    # --- Amount features ---
    df['log_amount']   = np.log1p(df['TransactionAmt'])
    df['amount_cents'] = (df['TransactionAmt'] % 1 == 0).astype(int)  # round amount flag

    # --- Card velocity proxy (card1 + addr1 aggregate) ---
    if 'card1' in df.columns and 'addr1' in df.columns:
        card_addr_key        = df['card1'].astype(str) + '_' + df['addr1'].astype(str)
        df['card_addr_freq'] = card_addr_key.map(card_addr_key.value_counts())

    # --- Email domain flags ---
    if 'P_emaildomain' in df.columns:
        risky_domains        = ['gmail.com', 'yahoo.com', 'hotmail.com']
        df['risky_email']    = df['P_emaildomain'].isin(risky_domains).astype(int)

    return df

df = engineer_features(df)
print(f'Features after engineering: {df.shape[1]}')
print('New features added: hour, day_of_week, is_weekend, is_night, log_amount, amount_cents, card_addr_freq, risky_email')

## 3. Missing Value Imputation

In [ ]:
# Drop columns with >80% missing — too noisy to impute reliably
missing_pct = df.isnull().sum() / len(df)
drop_cols   = missing_pct[missing_pct > 0.80].index.tolist()
df          = df.drop(columns=drop_cols)
print(f'Dropped {len(drop_cols)} columns with >80% missing')

# Numeric: median imputation
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c != 'isFraud']
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Categorical: mode imputation
cat_cols = df.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print(f'Remaining nulls: {df.isnull().sum().sum()}')

## 4. Categorical Encoding

In [ ]:
le = LabelEncoder()
for col in cat_cols:
    if col in df.columns:
        df[col] = le.fit_transform(df[col].astype(str))

print(f'Encoded {len(cat_cols)} categorical columns')
print(f'Final feature count: {df.shape[1] - 2}')  # exclude TransactionID + isFraud

## 5. Train / Validation / Test Split

In [ ]:
# Drop ID column
feature_cols = [c for c in df.columns if c not in ['TransactionID', 'isFraud']]
X = df[feature_cols]
y = df['isFraud']

# 70% train | 15% val | 15% test — stratified
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

print(f'Train : {X_train.shape[0]:,} rows | Fraud: {y_train.sum():,} ({y_train.mean()*100:.2f}%)')
print(f'Val   : {X_val.shape[0]:,}   rows | Fraud: {y_val.sum():,} ({y_val.mean()*100:.2f}%)')
print(f'Test  : {X_test.shape[0]:,}  rows | Fraud: {y_test.sum():,} ({y_test.mean()*100:.2f}%)')

## 6. SMOTE — Handle Class Imbalance on Training Set ONLY

In [ ]:
# ⚠️ CRITICAL: Apply SMOTE ONLY to training data
# Val and Test must remain at original distribution to reflect real-world performance
smote = SMOTE(sampling_strategy=0.20, k_neighbors=5, random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f'Before SMOTE | Train fraud rate: {y_train.mean()*100:.2f}%')
print(f'After  SMOTE | Train fraud rate: {y_train_bal.mean()*100:.2f}%')
print(f'Train size after SMOTE: {len(X_train_bal):,} rows')

# Feature scaling
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_bal)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

print('\nPreprocessing complete. Ready for modeling.')

## 7. Save Processed Data

In [ ]:
import os, joblib
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../outputs', exist_ok=True)

# Save splits
pd.DataFrame(X_train_bal, columns=feature_cols).to_csv('../data/processed/X_train.csv', index=False)
pd.Series(y_train_bal, name='isFraud').to_csv('../data/processed/y_train.csv', index=False)
pd.DataFrame(X_val, columns=feature_cols).to_csv('../data/processed/X_val.csv', index=False)
pd.Series(y_val, name='isFraud').to_csv('../data/processed/y_val.csv', index=False)
pd.DataFrame(X_test, columns=feature_cols).to_csv('../data/processed/X_test.csv', index=False)
pd.Series(y_test, name='isFraud').to_csv('../data/processed/y_test.csv', index=False)

# Save scaler + feature list
joblib.dump(scaler, '../outputs/scaler.pkl')
joblib.dump(feature_cols, '../outputs/feature_cols.pkl')

print('All processed files saved.')